# 03 - Capa Gold: Feature Engineering + Feature Store

Este notebook toma los datos validados de Silver y construye dos tablas de features
registradas en **Unity Catalog Feature Store**:

| Feature Table | Primary Key | Descripcion |
| --- | --- | --- |
| `cv_v2_gold_image_features` | `image_id` | Features agregadas por imagen |
| `cv_v2_gold_annotation_features` | `image_id`, `object_index` | Features por anotacion individual |

**Prerequisito:** Ejecutar notebooks `01_ingesta_bronze` y `02_silver_validacion`

In [0]:
%pip install databricks-feature-engineering -q

In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType

# -- Unity Catalog --
CATALOG = "jgworkspaceclassic_catalog"
SCHEMA = "infra_demo"

SILVER_TABLE = f"{CATALOG}.{SCHEMA}.cv_v2_silver"
GOLD_IMAGE_TABLE = f"{CATALOG}.{SCHEMA}.cv_v2_gold_image_features"
GOLD_ANNOTATION_TABLE = f"{CATALOG}.{SCHEMA}.cv_v2_gold_annotation_features"

print(f"Silver:          {SILVER_TABLE}")
print(f"Gold (imagenes): {GOLD_IMAGE_TABLE}")
print(f"Gold (anotac.):  {GOLD_ANNOTATION_TABLE}")

In [0]:
df_valid = spark.table(SILVER_TABLE).filter(F.col("is_valid"))
total_valid = df_valid.count()
print(f"Registros validos en Silver: {total_valid}")

In [0]:
df_image_features = (
    df_valid
    .groupBy("image_id", "image_path", "split")
    .agg(
        # Conteos
        F.count("*").alias("num_objects"),
        F.sum(F.when(F.col("class_name") == "Captain", 1).otherwise(0)).alias("num_captain"),
        F.sum(F.when(F.col("class_name") == "Gohan", 1).otherwise(0)).alias("num_gohan"),
        F.countDistinct("class_name").alias("num_distinct_classes"),

        # Estadisticas de bounding box
        F.avg("bbox_area").alias("avg_bbox_area"),
        F.min("bbox_area").alias("min_bbox_area"),
        F.max("bbox_area").alias("max_bbox_area"),
        F.stddev("bbox_area").alias("std_bbox_area"),
        F.sum("bbox_area").alias("total_bbox_coverage"),

        # Estadisticas de aspect ratio
        F.avg(
            F.when(F.col("bbox_height") > 0, F.col("bbox_width") / F.col("bbox_height"))
        ).alias("avg_aspect_ratio"),

        # Complejidad de poligonos
        F.avg("num_polygon_points").alias("avg_polygon_complexity"),
        F.max("num_polygon_points").alias("max_polygon_complexity"),

        # Tamano de imagen
        F.first("image_size_bytes").alias("image_size_bytes"),

        # Centroides promedio (donde se concentran los objetos)
        F.avg((F.col("bbox_x_min") + F.col("bbox_x_max")) / 2).alias("avg_centroid_x"),
        F.avg((F.col("bbox_y_min") + F.col("bbox_y_max")) / 2).alias("avg_centroid_y"),

        # Clases presentes
        F.collect_set("class_name").alias("classes_present"),
    )
    # Features derivadas
    .withColumn("has_both_classes",
        (F.col("num_captain") > 0) & (F.col("num_gohan") > 0))
    .withColumn("dominant_class",
        F.when(F.col("num_captain") > F.col("num_gohan"), "Captain")
         .when(F.col("num_gohan") > F.col("num_captain"), "Gohan")
         .otherwise("balanced"))
    .withColumn("class_balance_ratio",
        F.when(F.col("num_objects") > 0,
               F.least(F.col("num_captain"), F.col("num_gohan")).cast("float") /
               F.greatest(F.col("num_captain"), F.col("num_gohan")).cast("float"))
         .otherwise(F.lit(0.0)))
    # Densidad de objetos (total_coverage como proxy)
    .withColumn("object_density",
        F.when(F.col("total_bbox_coverage") > 1.0, F.lit(1.0))
         .otherwise(F.col("total_bbox_coverage")))
    .withColumn("computed_at", F.current_timestamp())
)

# Reemplazar NaN en std_bbox_area (imagenes con 1 solo objeto)
df_image_features = df_image_features.fillna(0.0, subset=["std_bbox_area"])

print(f"Imagenes con features: {df_image_features.count()}")
display(df_image_features.limit(10))

In [0]:
df_annotation_features = (
    df_valid
    .withColumn("bbox_width", F.col("bbox_x_max") - F.col("bbox_x_min"))
    .withColumn("bbox_height", F.col("bbox_y_max") - F.col("bbox_y_min"))
    .withColumn("bbox_area", F.col("bbox_width") * F.col("bbox_height"))
    .withColumn("bbox_aspect_ratio",
        F.when(F.col("bbox_height") > 0,
               F.col("bbox_width") / F.col("bbox_height"))
         .otherwise(F.lit(0.0)))
    .withColumn("bbox_perimeter",
        2 * (F.col("bbox_width") + F.col("bbox_height")))
    .withColumn("centroid_x",
        (F.col("bbox_x_min") + F.col("bbox_x_max")) / 2)
    .withColumn("centroid_y",
        (F.col("bbox_y_min") + F.col("bbox_y_max")) / 2)
    # Distancia del centroide al centro de la imagen
    .withColumn("distance_to_center",
        F.sqrt(
            F.pow(((F.col("bbox_x_min") + F.col("bbox_x_max")) / 2) - 0.5, 2) +
            F.pow(((F.col("bbox_y_min") + F.col("bbox_y_max")) / 2) - 0.5, 2)
        ))
    # Categoria de tamano
    .withColumn("size_category",
        F.when(F.col("bbox_area") < 0.01, "small")
         .when(F.col("bbox_area") < 0.05, "medium")
         .when(F.col("bbox_area") < 0.15, "large")
         .otherwise("very_large"))
    # Posicion relativa en la imagen (cuadrante)
    .withColumn("position_quadrant",
        F.when((F.col("centroid_x") < 0.5) & (F.col("centroid_y") < 0.5), "top_left")
         .when((F.col("centroid_x") >= 0.5) & (F.col("centroid_y") < 0.5), "top_right")
         .when((F.col("centroid_x") < 0.5) & (F.col("centroid_y") >= 0.5), "bottom_left")
         .otherwise("bottom_right"))
    # Forma del objeto
    .withColumn("shape_category",
        F.when(F.col("bbox_aspect_ratio") < 0.7, "tall")
         .when(F.col("bbox_aspect_ratio") > 1.4, "wide")
         .otherwise("square"))
    .select(
        "image_id", "object_index", "split", "class_id", "class_name",
        "bbox_x_min", "bbox_y_min", "bbox_x_max", "bbox_y_max",
        "bbox_width", "bbox_height", "bbox_area", "bbox_perimeter",
        "bbox_aspect_ratio", "centroid_x", "centroid_y",
        "distance_to_center", "num_polygon_points",
        "size_category", "position_quadrant", "shape_category",
    )
    .withColumn("computed_at", F.current_timestamp())
)

print(f"Anotaciones con features: {df_annotation_features.count()}")
display(df_annotation_features.limit(10))

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

# --- Feature Table: Imagenes ---
spark.sql(f"DROP TABLE IF EXISTS {GOLD_IMAGE_TABLE}")

fe.create_table(
    name=GOLD_IMAGE_TABLE,
    primary_keys=["image_id"],
    df=df_image_features,
    description=(
        "Features agregadas por imagen para deteccion de juguetes (Captain/Gohan). "
        "Incluye conteos de objetos, estadisticas de bounding box, complejidad de poligonos, "
        "balance de clases y densidad de objetos."
    ),
)
print(f"Feature table creada: {GOLD_IMAGE_TABLE}")

# --- Feature Table: Anotaciones ---
spark.sql(f"DROP TABLE IF EXISTS {GOLD_ANNOTATION_TABLE}")

fe.create_table(
    name=GOLD_ANNOTATION_TABLE,
    primary_keys=["image_id", "object_index"],
    df=df_annotation_features,
    description=(
        "Features por anotacion individual para deteccion de juguetes. "
        "Incluye geometria del bbox, aspect ratio, centroide, distancia al centro, "
        "categoria de tamano, cuadrante y forma del objeto."
    ),
)
print(f"Feature table creada: {GOLD_ANNOTATION_TABLE}")

In [0]:
%sql
-- Ver informacion de las tablas registradas como Feature Tables
DESCRIBE TABLE EXTENDED jgworkspaceclassic_catalog.infra_demo.cv_v2_gold_image_features;

In [0]:
print("=" * 70)
print("RESUMEN DEL PIPELINE MEDALLION v2")
print("=" * 70)

tables = [
    (f"{CATALOG}.{SCHEMA}.cv_v2_bronze", "Bronze"),
    (SILVER_TABLE, "Silver"),
    (GOLD_IMAGE_TABLE, "Gold (Image Features)"),
    (GOLD_ANNOTATION_TABLE, "Gold (Annotation Features)"),
]

for table_name, layer in tables:
    count = spark.table(table_name).count()
    print(f"  {layer:35s} | {table_name:60s} | {count:>6} rows")

print()
print("Feature Store tables registradas en Unity Catalog:")
print(f"  - {GOLD_IMAGE_TABLE}")
print(f"  - {GOLD_ANNOTATION_TABLE}")

In [0]:
display(
    spark.table(GOLD_IMAGE_TABLE)
    .groupBy("split")
    .agg(
        F.count("*").alias("imagenes"),
        F.avg("num_objects").cast("decimal(4,2)").alias("avg_objetos"),
        F.avg("avg_bbox_area").cast("decimal(6,4)").alias("avg_area"),
        F.avg("avg_polygon_complexity").cast("decimal(4,1)").alias("avg_complejidad"),
        F.sum(F.when(F.col("has_both_classes"), 1).otherwise(0)).alias("con_ambas_clases"),
    )
    .orderBy("split")
)